In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\bhara\OneDrive\Desktop\FINBRIDGE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../data/fiqa.csv')
print(df.shape)
df.head()

(14511, 2)


,question,answer
0,What do brokers do with bad stock?,"For every seller, there's a buyer. Buyers may ..."
1,Why do investors buy stock that had appreciated?,"You seem to prefer to trade like I do: ""Buy lo..."
2,Is it ever a good idea to close credit cards?,"Yes, it can be a good idea to close unused cre..."
3,If something is coming into my account will it...,The bank will make this even more confusing be...
4,What one bit of financial advice do you wish y...,When I was contracting I wish I had joined a t...


In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3285.40it/s]


Model loaded!


In [8]:
question_vectors = model.encode(df['question'].tolist(), show_progress_bar=True)
print("Shape:", question_vectors.shape)

Batches: 100%|██████████| 454/454 [01:02<00:00,  7.26it/s]


Shape: (14511, 384)


In [9]:
def find_answer(query, top_k=3):
    query_vector = model.encode([query])
    similarities = cosine_similarity(query_vector, question_vectors)
    top_indices = similarities[0].argsort()[-top_k:][::-1]
    
    print(f"Query: {query}\n")
    for i, idx in enumerate(top_indices):
        print(f"Result {i+1}:")
        print(f"Question: {df['question'][idx]}")
        print(f"Answer: {df['answer'][idx]}")
        print(f"Similarity Score: {similarities[0][idx]:.4f}")
        print("---")

In [10]:
find_answer("How should I invest my money in stocks?")

Query: How should I invest my money in stocks?

Result 1:
Question: How do I get into investing in stocks?
Answer: Start by paying down any high interest debt you may have, like credit cards.  Reason being that they ultimately eat into any (positive) returns you may have from investing.  Another good reason is to build up some discipline.  You will need discipline to be a successful investor. Educate yourself about investing. The Motley Fool is probably still a good place to start.  I would also suggest getting into the habit of reading the Wall Street Journal or at the very least the business section of the New York Times.  You'll be overwhelmed with the terminology at first, but stick with it.  It is certainly worth it, if you want to be an investor.  The Investor's Business Daily is another good resource for information, though you will be lost in the deep end of the pool with that publication for sure.  (That is not a reason to avoid getting familiar with it.  Though at first, it m

In [12]:
print("MODEL COMPARISON")
print("="*50)
print("TF-IDF Score:      0.5555 (keyword matching)")
print("Word2Vec Score:    0.9688 (word meaning)")
print("Transformer Score: 0.7903 (sentence meaning)")

MODEL COMPARISON
TF-IDF Score:      0.5555 (keyword matching)
Word2Vec Score:    0.9688 (word meaning)
Transformer Score: 0.7903 (sentence meaning)
